# Milestone 1 - Derrick Jaskiel

## Imports

In [12]:
from pathlib import Path
import requests
import zipfile
import os
import pandas as pd
from urllib.request import urlretrieve

In [11]:
import time, psutil, os, functools

_proc = psutil.Process(os.getpid())

def measure(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        m0 = _proc.memory_info().rss / 1e6
        c0 = time.process_time()
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        print(f"{fn.__name__}:  wall={time.perf_counter()-t0:.1f}s  "
              f"cpu={time.process_time()-c0:.1f}s  "
              f"mem Δ{_proc.memory_info().rss/1e6 - m0:+.0f} MB")
        return out
    return wrapper

## 1. Downloading the Data

In [19]:
ARTICLE_ID = 14096681
WORKDIR = Path.cwd()
OUTPUT_DIR = WORKDIR / "data"
OUTPUT_DIR.mkdir(exist_ok = True)
FORCE_DOWNLOAD = False

api_url = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"
article = requests.get(api_url).json()

In [20]:
files_to_dl = ["data.zip"]
for file in article["files"]:
    if file["name"] in files_to_dl:

        output_path = OUTPUT_DIR / file["name"]

        if output_path.exists() and not FORCE_DOWNLOAD:
            print(f"{file['name']} already exists. Skipping.")
        else:
            output_path.parent.mkdir(exist_ok = True)

            print(f"Downloading {file['name']}...")
            urlretrieve(file["download_url"], output_path)

In [21]:
zip_path = OUTPUT_DIR / "data.zip"
extract_dir = OUTPUT_DIR / "data"

if not extract_dir.exists():
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(OUTPUT_DIR)
else:
    print("Files already extracted. Skipping extraction.")

Extracting files...


In [18]:
print(zip_path.stat().st_size)

483393536


## 2. Combining Data CSVs

For the task of combining CSV's, I programmatically collected all CSV files in the data/ folder, excluded `observed_daily_rainfall_SYD.csv `, and extracted the climate model name from each filename, appendeding it as a new model column. I then concatenated all model files into one dataframe and saved the result as a single combined CSV.

### Reflection

| Member | Approach | OS | RAM | Processor | SSD? | wall (s) | cpu (s) | mem Δ (MB) |
|:------:|:--------:|:--:|:---:|:---------:|:----:|:--------:|:-------:|:----------:|
| 1 (Me) | combine | Windows 11 version 25H2 | 16GB | AMD Ryzen 7 5825U | Yes | 406.9 | 392.8 | +5225 |
| 2 | combine | MacOS Tahoe 26.3.1 (a) | 24GB | Apple M4 Pro | Yes | 149.0 | 148.4 | -2327 |
| 3 | combine | MacOS Sequoia 15.4 | 16GB | M1 | Yes | 401.9 | 392.5 | +560 |

The Mac M4 Pro machine produced the strongest performance, with significantly lower wall and CPU times (~149s) than the Windows and M1 machines (~400s). This exemplifies how newer hardware and higher memory capacity substantially improve data processing speed. For my machine and the M1 system, wall time and CPU time were very close, suggesting that the task was primarily CPU-bound. The large positive memory increase on my machine indicates that loading and concatenating all dataframes together is memory intensive, and the negative memory delta with the M4 Pro likely reflects more efficient memory management.


In [23]:
def extract_model_name(filename):
    return filename.split("_")[0]

@measure
def combine_csvs(folder, output_file = "combined_data.csv"):
    folder = Path(folder)

    csv_files = sorted(
        f for f in folder.glob("*.csv")
        if f.name != "observed_daily_rainfall_SYD.csv"
    )


    dfs = []

    for file in csv_files:
        print(f"Reading {file.name}...")
        df = pd.read_csv(file)
        df["model"] = extract_model_name(file.name)
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(output_file, index=False)

    print(f"\nCombined shape: {combined_df.shape}")
    print(f"Saved to: {output_file}")

    return combined_df

In [24]:
combined_df = combine_csvs("data", output_file = "data/combined_data.csv")
combined_df.head()
combined_df["model"].value_counts()
combined_df.shape
combined_df.columns

Reading ACCESS-CM2_daily_rainfall_NSW.csv...
Reading ACCESS-ESM1-5_daily_rainfall_NSW.csv...
Reading AWI-ESM-1-1-LR_daily_rainfall_NSW.csv...
Reading BCC-CSM2-MR_daily_rainfall_NSW.csv...
Reading BCC-ESM1_daily_rainfall_NSW.csv...
Reading CanESM5_daily_rainfall_NSW.csv...
Reading CMCC-CM2-HR4_daily_rainfall_NSW.csv...
Reading CMCC-CM2-SR5_daily_rainfall_NSW.csv...
Reading CMCC-ESM2_daily_rainfall_NSW.csv...
Reading EC-Earth3-Veg-LR_daily_rainfall_NSW.csv...
Reading FGOALS-f3-L_daily_rainfall_NSW.csv...
Reading FGOALS-g3_daily_rainfall_NSW.csv...
Reading GFDL-CM4_daily_rainfall_NSW.csv...
Reading GFDL-ESM4_daily_rainfall_NSW.csv...
Reading INM-CM4-8_daily_rainfall_NSW.csv...
Reading INM-CM5-0_daily_rainfall_NSW.csv...
Reading KIOST-ESM_daily_rainfall_NSW.csv...
Reading MIROC6_daily_rainfall_NSW.csv...
Reading MPI-ESM-1-2-HAM_daily_rainfall_NSW.csv...
Reading MPI-ESM1-2-HR_daily_rainfall_NSW.csv...
Reading MPI-ESM1-2-LR_daily_rainfall_NSW.csv...
Reading MRI-ESM2-0_daily_rainfall_NSW.csv.

Index(['time', 'lat_min', 'lat_max', 'lon_min', 'lon_max', 'rain (mm/day)',
       'model'],
      dtype='object')

## 3. Load to Memory + Python EDA


### Reflection

| Member | Approach | OS | RAM | Processor | SSD? | wall (s) | cpu (s) | mem Δ (MB) |
|:------:|:--------:|:--:|:---:|:---------:|:----:|:--------:|:-------:|:----------:|
| 1 (Me) | usecols | Windows 11 version 25H2 | 16GB | AMD Ryzen 7 5825U | Yes | 55.0 | 53.0 | +3 |
| 2 | usecols | MacOS Tahoe 26.3.1 (a) | 24GB | Apple M4 Pro | Yes | 19.2 | 19.1 | -752 |
| 3 | usecols | MacOS Sequoia 15.4 | 16GB | M1 | Yes | x | x | x |
| 4 | usecols | x | x | x | x | x | x | x |
| 1 (Me) | dtypes | Windows 11 version 25H2 | 16GB | AMD Ryzen 7 5825U | Yes | 47.8 | 46.2 | +1 |
| 2 | dtypes | MacOS Tahoe 26.3.1 (a) | 24GB | Apple M4 Pro | Yes | 27.2 | 27.1 | +1530 |
| 3 | dtypes | MacOS Sequoia 15.4 | 16GB | M1 | Yes | x | x | x |
| 4 | dtypes | x | x | x | x | x | x | x |

### Approach 1 (usecols)

This approach requires that we only read in the columns we explicitly need, directly reducing memory usage by excluding unnecessary data.

In [28]:
combined_csv_path = "data/combined_data.csv"

sample = pd.read_csv(combined_csv_path, nrows = 5)
sample.head()
sample.columns.tolist()

['time', 'lat_min', 'lat_max', 'lon_min', 'lon_max', 'rain (mm/day)', 'model']

In [30]:
@measure
def eda_usecols(csv_path):
    needed_cols = ["model", "rain (mm/day)"]
    
    df = pd.read_csv(csv_path, usecols = needed_cols)
    
    results = {
        "n_rows": len(df),
        "model_counts": df["model"].value_counts(),
        "rainfall_summary": df["rain (mm/day)"].describe(),
        "missing_values": df.isna().sum()
    }
    return results

In [31]:
result_usecols = eda_usecols(combined_csv_path)

print("Rows:", result_usecols["n_rows"])
print("\nModel counts:\n", result_usecols["model_counts"])
print("\nRainfall summary:\n", result_usecols["rainfall_summary"])
print("\nMissing values:\n", result_usecols["missing_values"])

eda_usecols:  wall=55.0s  cpu=53.0s  mem Δ+3 MB
Rows: 62467843

Model counts:
 model
MPI-ESM1-2-HR       5154240
CMCC-CM2-SR5        3541230
CMCC-ESM2           3541230
NorESM2-MM          3541230
TaiESM1             3541230
CMCC-CM2-HR4        3541230
SAM0-UNICON         3541153
FGOALS-f3-L         3219300
GFDL-CM4            3219300
GFDL-ESM4           3219300
EC-Earth3-Veg-LR    3037320
MRI-ESM2-0          3037320
BCC-CSM2-MR         3035340
MIROC6              2070900
ACCESS-CM2          1932840
ACCESS-ESM1-5       1610700
INM-CM4-8           1609650
INM-CM5-0           1609650
FGOALS-g3           1287720
KIOST-ESM           1287720
MPI-ESM-1-2-HAM      966420
AWI-ESM-1-1-LR       966420
MPI-ESM1-2-LR        966420
NESM3                966420
NorESM2-LM           919800
BCC-ESM1             551880
CanESM5              551880
Name: count, dtype: int64

Rainfall summary:
 count    5.924854e+07
mean     1.901170e+00
std      5.585735e+00
min     -3.807373e-12
25%      3.838413e-06
50%

### Approach 2 (dtype)

Rather than removing data like we did in the previous approach, here we change the way the it is represented, implementing more memory-efficient representations for repeated strings and numeric columns.

In [32]:
@measure
def eda_dtype(csv_path):
    dtype_map = {
        "model": "category",
        "rainfall": "float32"
    }
    
    df = pd.read_csv(csv_path, usecols=["model", "rain (mm/day)"], dtype=dtype_map)
    
    results = {
        "n_rows": len(df),
        "model_counts": df["model"].value_counts(),
        "rainfall_summary": df["rain (mm/day)"].describe(),
        "missing_values": df.isna().sum()
    }
    return results

In [33]:
result_dtype = eda_dtype(combined_csv_path)

print("Rows:", result_dtype["n_rows"])
print("\nModel counts:\n", result_dtype["model_counts"])
print("\nRainfall summary:\n", result_dtype["rainfall_summary"])
print("\nMissing values:\n", result_dtype["missing_values"])

eda_dtype:  wall=47.8s  cpu=46.2s  mem Δ+1 MB
Rows: 62467843

Model counts:
 model
MPI-ESM1-2-HR       5154240
CMCC-CM2-SR5        3541230
CMCC-ESM2           3541230
NorESM2-MM          3541230
TaiESM1             3541230
CMCC-CM2-HR4        3541230
SAM0-UNICON         3541153
FGOALS-f3-L         3219300
GFDL-CM4            3219300
GFDL-ESM4           3219300
EC-Earth3-Veg-LR    3037320
MRI-ESM2-0          3037320
BCC-CSM2-MR         3035340
MIROC6              2070900
ACCESS-CM2          1932840
ACCESS-ESM1-5       1610700
INM-CM4-8           1609650
INM-CM5-0           1609650
FGOALS-g3           1287720
KIOST-ESM           1287720
MPI-ESM-1-2-HAM      966420
AWI-ESM-1-1-LR       966420
MPI-ESM1-2-LR        966420
NESM3                966420
NorESM2-LM           919800
BCC-ESM1             551880
CanESM5              551880
Name: count, dtype: int64

Rainfall summary:
 count    5.924854e+07
mean     1.901170e+00
std      5.585735e+00
min     -3.807373e-12
25%      3.838413e-06
50%  

## 4. R EDA